# 17_product_search: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/17_product_search/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Separate search intent from exact eligibility rules.

Run: uv run  python modules/17_product_search/examples/01_from_scratch.py
Expected: three typed clauses; ambiguous alternatives produce a warning.
CPU: seconds to one minute. Needs: base environment. No implicit downloads.
"""

from dataclasses import asdict

from common.search.query_parser import parse_query
from common.stores.filters import matches, parse_filter


def main() -> None:
    parsed = parse_query("mug warmer under $100, in stock, in the north region")
    print(asdict(parsed))
    # KEY IDEA: filter comparisons have boundaries that similarity cannot enforce.
    eligible = matches({"price": 99.99, "stock": 1, "region": "north"}, parse_filter(parsed.filter))
    boundary = matches({"price": 100, "stock": 1, "region": "north"}, parse_filter(parsed.filter))
    assert eligible and not boundary
    print(parse_query("mug under $20 or over $100").warnings)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/17_product_search/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None:
    print("SKIP native comparison: --extra qdrant")
else:
    """Run filtered hybrid search in a real local Qdrant database.

    Run: uv run --extra qdrant python modules/17_product_search/examples/02_production.py
    Expected: five eligible products and native hybrid results.
    CPU: seconds to one minute. Needs: qdrant extra, committed catalog. No implicit downloads.
    """
    # course-extra: qdrant

    from common.apps.database import AppDatabase
    from common.apps.product_search import ProductSearch
    from common.embeddings import frozen_query
    from common.pretty import show_hits
    from common.search.query_parser import parse_query
    from common.settings import data_dir

    def main() -> None:
        query, meta = frozen_query("q000", data_dir())
        text = "mug warmer under $100 in stock in the north region"
        with AppDatabase("qdrant-local") as db:
            search = ProductSearch(db, 500)
            result = search.search(text, query, k=5)
            show_hits(
                result.hits, result.timings_ms["retrieval"], title="Application BM25 + dense RRF"
            )
            # KEY IDEA: native fusion and application fusion are separate implementations.
            native = search.store.hybrid_search(
                "mug warmer", query, 5, parse_query(text).filter, space=meta["space"]
            )
            show_hits(native, 0, title="Native Qdrant sparse-count + dense RRF (untimed)")
            assert all(h.metadata["price"] < 100 for h in native)

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/17_product_search/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compare dense and hybrid SKU relevance on the same records.

Run: uv run  python modules/17_product_search/examples/03_measure.py
Expected: hybrid retrieves the literal SKU; timings and nDCG are recorded.
CPU: seconds to one minute. Needs: base, saved vectors. No implicit downloads.
"""

from common.apps.database import AppDatabase
from common.apps.product_search import ProductSearch
from common.bench.metrics import ndcg_at_k
from common.data import search_queries
from common.embeddings import frozen_query
from common.settings import data_dir
from common.teaching import evidence


def main() -> None:
    q = next(q for q in search_queries() if q["id"] == "q001")
    vector, _ = frozen_query(q["id"], data_dir())
    rows = {}
    with AppDatabase() as db:
        search = ProductSearch(db, 1000)
        for mode in ["dense", "hybrid", "lexical"]:
            result = search.search(q["text"], vector, mode=mode)
            ids = [h.id for h in result.hits]
            # KEY IDEA: relevance comes from independent judgments, not cosine scores.
            rows[mode] = {
                "ids": ids,
                "ndcg10": ndcg_at_k(ids, q["qrels"], 10),
                "timings_ms": result.timings_ms,
            }
    print(rows)
    evidence("product-ranking", rows, tier=4)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 04_reranking

Source: `modules/17_product_search/examples/04_reranking.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("sentence_transformers") is None:
    print("SKIP native comparison: --extra models")
else:
    """Observe candidate reranking and protect an exact identifier.

    Run: uv run --extra models python modules/17_product_search/examples/04_reranking.py
    Expected: query/document scores and an exact SKU pinned first.
    CPU: seconds to one minute. Needs: models extra, cached cross encoder. No implicit downloads.
    """
    # course-extra: models

    from common.apps.database import AppDatabase
    from common.apps.product_search import ProductSearch
    from common.embeddings import Reranker, frozen_query
    from common.pretty import show_hits
    from common.settings import data_dir

    def main() -> None:
        vector, _ = frozen_query("q001", data_dir())
        with AppDatabase() as db:
            search = ProductSearch(db, 100)
            # KEY IDEA: a cross encoder cannot recover a document absent from candidates.
            result = search.search("MM-11-0000031", vector, k=5, reranker=Reranker())
            show_hits(
                result.hits,
                sum(result.timings_ms.values()),
                title="Reranking with identifier policy",
            )
            assert result.hits[0].metadata["sku"] == "MM-11-0000031"

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 05_multilingual

Source: `modules/17_product_search/examples/05_multilingual.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compare real Qwen and MiniLM embeddings on a matched 500-row corpus.

Run: uv run  python modules/17_product_search/examples/05_multilingual.py
Expected: separate held-out nDCG for each model.
CPU: seconds to one minute. Needs: prepared Qwen vectors, base. No implicit downloads.
"""

from common.apps.database import AppDatabase
from common.apps.evaluation import multilingual_evaluation
from common.teaching import evidence


def main() -> None:
    with AppDatabase() as db:
        # KEY IDEA: re-embed both documents and queries when changing model space.
        result = multilingual_evaluation(db)
    print(result["held_out"])
    evidence("multilingual", result, tier=4)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.search.query_parser import parse_query

print(parse_query("chair at most $100 in stock"))
# Change at most to under. Which boundary changes?

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.